<a href="https://colab.research.google.com/github/Haseeb-zai30/ARKANOID-SFML-/blob/main/day_10_Support_Vector_Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Support Vector Machine vs. Logistic Regression
## A Complete End-to-End Practical Guide

In this notebook, we will put the theory from the blackboard into practice. We will:
1. Load a real dataset.
2. Preprocess the data (Crucial for SVM!).
3. Train both an SVM and a Logistic Regression model.
4. Evaluate them using robust metrics (Accuracy, Precision, Recall, F1) and Confusion Matrices.
5. Visually plot the Decision Boundary, the Margins, and the Support Vectors.
6. Final Exercise: Apply this to a new dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

### 1. Data Accession

We will be using the **Social Network Ads** dataset. It is a very popular 2-dimensional dataset found on Kaggle, making it perfect for plotting decision boundaries visually.

**Scenario:** Predicting whether a user will purchase a product based on their `Age` and `EstimatedSalary`.

*Note: To make this notebook run out-of-the-box, we are fetching the raw CSV directly from a public repository that mirrors the Kaggle dataset.*

In [ ]:
# Load the dataset directly via URL
url = 'https://raw.githubusercontent.com/shivang98/Social-Network-ads-Boost/master/Social_Network_Ads.csv'
df = pd.read_csv(url)

# Display the first 5 rows
df.head()

### 2. Data Preprocessing (Detailed)

SVMs are **highly sensitive to unscaled data**. Because SVM calculates the geometric distance between points, if `EstimatedSalary` ranges from 15,000 to 150,000, and `Age` ranges from 18 to 60, the salary feature will completely dominate the distance calculations.

We must use `StandardScaler` to force both features to have a mean of 0 and a variance of 1.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step A: Feature Selection
# We select Age and EstimatedSalary as our X features, and Purchased as our target Y.
X = df[['Age', 'EstimatedSalary']].values
y = df['Purchased'].values

# Step B: Train/Test Split
# We split 75% for training and 25% for testing unseen data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Step C: Feature Scaling (Crucial for SVM!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Learn scaling parameters AND transform
X_test_scaled = scaler.transform(X_test)       # Only transform using learned parameters

print(f"Original Age range: {X_train[:, 0].min()} to {X_train[:, 0].max()}")
print(f"Scaled Age range: {X_train_scaled[:, 0].min():.2f} to {X_train_scaled[:, 0].max():.2f}")

### 3. Model Selection & Training

Let's train both models so we can compare them. We will use a **Linear Kernel** for the SVM to see a straight-line decision boundary similar to our blackboard examples.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Train Logistic Regression
logreg_model = LogisticRegression(random_state=42)
logreg_model.fit(X_train_scaled, y_train)

# Train Support Vector Machine
# kernel='linear' gives a straight line.
# C=1.0 is our regularization (Soft Margin strictness).
svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)

print("Both models trained successfully!")

### 4. Evaluation & Performance Metrics

Accuracy isn't everything. We will calculate Accuracy, Precision, Recall, and the F1-Score to get a holistic view of how our models are performing.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}\n")
    return y_pred

y_pred_logreg = evaluate_model("Logistic Regression", logreg_model, X_test_scaled, y_test)
y_pred_svm = evaluate_model("Support Vector Machine", svm_model, X_test_scaled, y_test)

In [ ]:
# Plotting the Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_logreg), annot=True, cmap='Blues', fmt='d', ax=axes[0])
axes[0].set_title('Logistic Regression - Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, cmap='Greens', fmt='d', ax=axes[1])
axes[1].set_title('SVM - Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

### 5. Visualizing the Math: Decision Boundary, Margins, and Support Vectors

This is where the geometry we discussed on the blackboard comes to life. We will plot the actual hyperplane (decision boundary), calculate the margin widths using our weight vector `w`, and circle the data points acting as our Support Vectors.

In [ ]:
def plot_svm_boundary(model, X, y):
    plt.figure(figsize=(12, 8))

    # 1. Create a meshgrid to plot the decision surface
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

    # Predict on the entire meshgrid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot the background color mapping
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')

    # Plot the original data points
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', s=50)

    # 2. Extract weights (w) and bias (b) from the model to draw the lines
    w = model.coef_[0]
    b = model.intercept_[0]

    # Formula for hyperplane: w0*x0 + w1*x1 + b = 0  =>  x1 = -(w0/w1)*x0 - (b/w1)
    a = -w[0] / w[1]
    xx_line = np.linspace(x_min, x_max)
    yy_hyperplane = a * xx_line - (b) / w[1]

    # Calculate the margin size: 1 / ||w||
    margin = 1 / np.sqrt(np.sum(model.coef_ ** 2))

    # Offset the hyperplane by the margin to get the dashed boundary lines
    yy_margin_up = yy_hyperplane + np.sqrt(1 + a ** 2) * margin
    yy_margin_down = yy_hyperplane - np.sqrt(1 + a ** 2) * margin

    # 3. Plot the Hyperplane and Margins
    plt.plot(xx_line, yy_hyperplane, 'k-', linewidth=2, label='Hyperplane (Decision Boundary)')
    plt.plot(xx_line, yy_margin_up, 'k--', linewidth=1.5, label='Positive Margin')
    plt.plot(xx_line, yy_margin_down, 'k--', linewidth=1.5, label='Negative Margin')

    # 4. Highlight the Support Vectors
    support_vectors = model.support_vectors_
    plt.scatter(support_vectors[:, 0], support_vectors[:, 1], s=200,
                linewidth=2, facecolors='none', edgecolors='yellow', label='Support Vectors')

    # Formatting the plot
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.title('SVM Decision Boundary, Margins, and Support Vectors (C=1.0)', fontsize=16)
    plt.xlabel('Age (Standardized)', fontsize=12)
    plt.ylabel('Estimated Salary (Standardized)', fontsize=12)
    plt.legend(loc='upper right')
    plt.show()

# Plot using the Training Set
plot_svm_boundary(svm_model, X_train_scaled, y_train)

### Student Exercise: Apply this to a complex Medical Dataset!

Now it is your turn. Head over to Kaggle and search for the **Heart Disease UCI** dataset (or the Heart Failure Prediction dataset).

**Your Tasks:**
1. **Data Accession:** Download the CSV and load it into a new DataFrame.
2. **Detailed Preprocessing:**
   - Check for missing/null values.
   - Separate your features `X` and target `y` (target is usually `target` or `HeartDisease`).
   - Use `pd.get_dummies()` or `LabelEncoder` if there are categorical text columns.
   - *Crucial:* Use `StandardScaler` to scale continuous features like `Cholesterol`, `RestingBP`, and `MaxHR`.
3. **Model Selection & Tuning:**
   - Train a Logistic Regression model.
   - Train an SVM model. *Experiment:* Try changing `kernel='linear'` to `kernel='rbf'`. Try changing the strictness parameter `C` from `0.1` to `100`. How does the accuracy change?
4. **Evaluation:**
   - Print the Accuracy, Precision, Recall, and F1-Score.
   - Plot the confusion matrix for both.
   - **Discussion Question:** In a medical dataset predicting heart disease, is it worse to have a *False Positive* (telling a healthy person they are sick) or a *False Negative* (telling a sick person they are healthy)? Based on your answer, which metric (Precision or Recall) should you care about the most in this dataset?